In [ ]:
# FIX CRITICO: Python 3.14 mudou o default de multiprocessing para 'forkserver'
# Isso causa OSError: Cannot allocate memory no WSL2 com num_workers > 0
# Forcar 'fork' resolve definitivamente o problema
import multiprocessing
multiprocessing.set_start_method('fork', force=True)
print(f'Metodo de multiprocessing: {multiprocessing.get_start_method()}')


In [ ]:
import os
import sys
import platform

print(f"Python version: {sys.version}")
print(f"OS: {platform.platform()}")
print(f"CPU Cores: {os.cpu_count()}")

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

    # Habilitar o modo de alta performance (TF32) nas arquiteturas Ampere/Hopper
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True # Otimiza convs para tamanhos estáticos

In [ ]:
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # torch.backends.cudnn.deterministic = True  # Desativado para não perder performance extrema

set_seed(42)
print("Global seed set to 42.")

In [ ]:
# Diretório local para salvar o CIFAR-100
DATA_PATH = "./data"
print("Usando dataset em: " + DATA_PATH)


In [ ]:
import torchvision
import torchvision.transforms as transforms
import torch

# Normalizacao padrao para CIFAR-100
mean = (0.5071, 0.4867, 0.4408)
std  = (0.2675, 0.2565, 0.2761)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

DATA_PATH = "."
train_dataset = torchvision.datasets.CIFAR100(root=DATA_PATH, train=True, download=True, transform=transform_train)
test_dataset  = torchvision.datasets.CIFAR100(root=DATA_PATH, train=False, download=True, transform=transform_val)

print(f"Train size: {len(train_dataset)}")
print(f"Test size:  {len(test_dataset)}")


In [ ]:
BATCH_SIZE = 256
NUM_WORKERS = 6  # Aumentado: 12 cores disponíveis

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=True, prefetch_factor=4  # Aumentado: imagens 32x32 são leves
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=4
)

print(f"DataLoaders otimizados criados com Batch Size = {BATCH_SIZE} e Workers = {NUM_WORKERS}")


In [ ]:
def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    # uniform
    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class BasicBlock(nn.Module):
    def __init__(self, in_planes, out_planes, stride, dropRate=0.0):
        super(BasicBlock, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_planes, out_planes, kernel_size=3, stride=1,
                               padding=1, bias=False)
        self.droprate = dropRate
        self.equalInOut = (in_planes == out_planes)
        self.convShortcut = (not self.equalInOut) and nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride,
                               padding=0, bias=False) or None

    def forward(self, x):
        if not self.equalInOut:
            x = self.relu1(self.bn1(x))
        else:
            out = self.relu1(self.bn1(x))
        out = self.relu2(self.bn2(self.conv1(out if self.equalInOut else x)))
        if self.droprate > 0:
            out = F.dropout(out, p=self.droprate, training=self.training)
        out = self.conv2(out)
        return torch.add(x if self.equalInOut else self.convShortcut(x), out)

class NetworkBlock(nn.Module):
    def __init__(self, nb_layers, in_planes, out_planes, block, stride, dropRate=0.0):
        super(NetworkBlock, self).__init__()
        self.layer = self._make_layer(block, in_planes, out_planes, nb_layers, stride, dropRate)
    def _make_layer(self, block, in_planes, out_planes, nb_layers, stride, dropRate):
        layers = []
        for i in range(nb_layers):
            layers.append(block(i == 0 and in_planes or out_planes, out_planes, i == 0 and stride or 1, dropRate))
        return nn.Sequential(*layers)
    def forward(self, x):
        return self.layer(x)

class WideResNet(nn.Module):
    def __init__(self, depth, num_classes, widen_factor=1, dropRate=0.0):
        super(WideResNet, self).__init__()
        nChannels = [16, 16*widen_factor, 32*widen_factor, 64*widen_factor]
        assert (depth - 4) % 6 == 0, 'depth should be 6n+4'
        n = (depth - 4) // 6
        block = BasicBlock
        # 1st conv before any network block
        self.conv1 = nn.Conv2d(3, nChannels[0], kernel_size=3, stride=1,
                               padding=1, bias=False)
        # 1st block
        self.block1 = NetworkBlock(n, nChannels[0], nChannels[1], block, 1, dropRate)
        # 2nd block
        self.block2 = NetworkBlock(n, nChannels[1], nChannels[2], block, 2, dropRate)
        # 3rd block
        self.block3 = NetworkBlock(n, nChannels[2], nChannels[3], block, 2, dropRate)
        # global average pooling and classifier
        self.bn1 = nn.BatchNorm2d(nChannels[3])
        self.relu = nn.ReLU(inplace=True)
        self.fc = nn.Linear(nChannels[3], num_classes)
        self.nChannels = nChannels[3]

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                m.bias.data.zero_()

    def forward(self, x):
        out = self.conv1(x)
        out = self.block1(out)
        out = self.block2(out)
        out = self.block3(out)
        out = self.relu(self.bn1(out))
        out = F.avg_pool2d(out, 8)
        out = out.view(-1, self.nChannels)
        return self.fc(out)

model = WideResNet(depth=28, num_classes=100, widen_factor=10, dropRate=0.3)
model = model.to(device)
model = model.to(memory_format=torch.channels_last)  # OPT: layout NHWC - melhor para convoluções na RTX 5060
model = torch.compile(model)  # Triton otimizado - max-autotune removido (RTX 5060 nao suporta)

# Total de parâmetros deve ficar em torno de 36.5M
print(f"Total parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

In [ ]:
EPOCHS = 300
LEARNING_RATE = 0.15

optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=5e-4, nesterov=True)

steps_per_epoch = len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    pct_start=0.1,
    anneal_strategy='cos',
    div_factor=10.0,
    final_div_factor=1e4
)

In [ ]:
def run_training():
    import time
    from torch.amp import autocast, GradScaler
    from torchmetrics import Accuracy
    import numpy as np

    scaler = GradScaler('cuda')
    accuracy_metric = Accuracy(task="multiclass", num_classes=100).to(device)
    top5_accuracy_metric = Accuracy(task="multiclass", num_classes=100, top_k=5).to(device)

    best_acc = 0.0
    cutmix_prob = 0.5

    print("Iniciando Treinamento...")
    for epoch in range(EPOCHS):
        model.train()

        train_loss_tensor = torch.zeros(1, device=device)
        correct_tensor = torch.zeros(1, device=device)
        total_tensor = torch.zeros(1, device=device)

        start_time = time.time()
        for batch_idx, (inputs, targets) in enumerate(train_loader):
            # OPT: channels_last para aproveitar kernels NHWC da RTX 5060
            inputs = inputs.to(device, non_blocking=True, memory_format=torch.channels_last)
            targets = targets.to(device, non_blocking=True)

            if torch.rand(1).item() < cutmix_prob:
                alpha = torch.tensor(1.0, device=device)
                lam = torch.distributions.Beta(alpha, alpha).sample().item()
                rand_index = torch.randperm(inputs.size(0), device=device)
                target_a = targets
                target_b = targets[rand_index]

                W, H = inputs.size(2), inputs.size(3)
                cut_rat = (1. - lam) ** 0.5
                cut_w = int(W * cut_rat)
                cut_h = int(H * cut_rat)
                cx = torch.randint(W, (1,)).item()
                cy = torch.randint(H, (1,)).item()
                bbx1 = max(cx - cut_w // 2, 0)
                bby1 = max(cy - cut_h // 2, 0)
                bbx2 = min(cx + cut_w // 2, W)
                bby2 = min(cy + cut_h // 2, H)

                inputs[:, :, bbx1:bbx2, bby1:bby2] = inputs[rand_index, :, bbx1:bbx2, bby1:bby2]
                lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (W * H))

                optimizer.zero_grad(set_to_none=True)
                with autocast('cuda'):
                    outputs = model(inputs)
                    loss = criterion(outputs, target_a) * lam + criterion(outputs, target_b) * (1. - lam)
            else:
                optimizer.zero_grad(set_to_none=True)
                with autocast('cuda'):
                    outputs = model(inputs)
                    loss = criterion(outputs, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            train_loss_tensor += loss.detach()
            _, predicted = outputs.max(1)
            correct_tensor += predicted.eq(targets).sum()
            total_tensor += targets.size(0)

        train_loss = train_loss_tensor.item() / len(train_loader)
        train_acc = 100. * correct_tensor.item() / total_tensor.item()

        # --- Validacao ---
        model.eval()
        # OPT: val_loss acumulado na GPU - zero sincronizacoes dentro do loop
        val_loss_tensor = torch.zeros(1, device=device)
        accuracy_metric.reset()
        top5_accuracy_metric.reset()

        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs = inputs.to(device, non_blocking=True, memory_format=torch.channels_last)
                targets = targets.to(device, non_blocking=True)
                with autocast('cuda'):
                    outputs = model(inputs)
                    loss = criterion(outputs, targets)

                # OPT: acumula na GPU, sem .item() no loop
                val_loss_tensor += loss.detach()
                accuracy_metric.update(outputs, targets)
                top5_accuracy_metric.update(outputs, targets)

        # Apenas UMA sincronizacao CPU-GPU por epoca
        val_loss = val_loss_tensor.item() / len(test_loader)
        val_acc = accuracy_metric.compute().item() * 100
        val_top5 = top5_accuracy_metric.compute().item() * 100

        epoch_time = time.time() - start_time
        print(f'Epoch [{epoch+1}/{EPOCHS}] | Time: {epoch_time:.0f}s | '
              f'Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | '
              f'Val Loss: {val_loss:.4f} | '
              f'Val Acc: {val_acc:.2f}% | Val Top5: {val_top5:.2f}% | '
              f'LR: {scheduler.get_last_lr()[0]:.5f}')

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), 'best_wrn28_10_cifar100.pth')
            print(f'>>> Melhor modelo salvo! Acuracia: {best_acc:.2f}%')

if __name__ == '__main__':
    run_training()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from torchmetrics import ConfusionMatrix

# Carregar o melhor modelo
model.load_state_dict(torch.load('best_wrn28_10_cifar100.pth'))
model.eval()

confmat = ConfusionMatrix(task="multiclass", num_classes=100).to(device)
accuracy_metric.reset()
top5_accuracy_metric.reset()

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        confmat.update(outputs, targets)
        accuracy_metric.update(outputs, targets)
        top5_accuracy_metric.update(outputs, targets)

cm = confmat.compute().cpu().numpy()
final_acc = accuracy_metric.compute().item() * 100
final_top5 = top5_accuracy_metric.compute().item() * 100

print(f"\n--- RESULTADOS FINAIS ---")
print(f"Melhor Top-1 Accuracy: {final_acc:.2f}%")
print(f"Melhor Top-5 Accuracy: {final_top5:.2f}%")

plt.figure(figsize=(20, 20))
sns.heatmap(cm, cmap='Blues', annot=False, xticklabels=False, yticklabels=False)
plt.title('Confusion Matrix - CIFAR-100', fontsize=16)
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()